In [2]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader("../data/Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splitted_documents = text_splitter.split_documents(documents)

In [4]:
from langchain_ollama import OllamaEmbeddings

embedding_model = OllamaEmbeddings(
    model="bge-m3:latest",
)

In [6]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(splitted_documents, embedding_model)

In [7]:
# 로컬 저장
vectorstore.save_local("./faiss_index")

In [8]:
vectorstore

In [9]:
vectorstore = None

In [10]:
vectorstore

In [11]:
# 벡터스토어 재로딩
vectorstore = FAISS.load_local(
    "./faiss_index", # 저장된 FAISS 인덱스 폴더의 경로
    embedding_model,
    allow_dangerous_deserialization=True, #  FAISS 인덱스 내 데이터 역직렬화(deserialization) 허용(신뢰할 수 있는 파일 일 경우)
)

In [12]:
vectorstore

In [13]:
query = "결혼하면 얼마 받을 수 있을까?"

In [14]:
results = vectorstore.similarity_search(query, k=3)

In [15]:
results

[Document(id='8fe8f2c0-b7bc-47b2-87c0-e3d157c0fd34', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 11, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'ModDate': "D:20260116002528+09'00'", 'Producer': 'Microsoft: Print To PDF', 'Title': 'Microsoft Word - Employee_Benefits_Guide_2026_v1.docx'}, page_content='구분 대상 휴가 (일수) 경조금 (만원) 화환/조화\n결혼 본인 5 일 100 + 화환 지원\n자녀 1 일 50 + 화환 지원\n30 -\n형제/자매 1 일\n30 -\n회갑/칠순 본인/배우자 부모 1 일\n출산 본인 출산휴가 (90 일) 출산 축하금 50 과일 바구니\n배우자 10 일 (유급) 출산 축하금 50 과일 바구니\n-\n사망 본인/배우자 500 + 장례용품 3 단 조화 + 근조기\n부모/배우자 부모 5 일 100 + 장례용품 3 단 조화 + 근조기\n30\n조부모/외조부모 3 일 조화\n30\n형제/자매 3 일 조화'),
 Document(id='54e7c807-6559-4592-91e0-63904ba906e9', metadata={'source': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'file_path': '../data/Employee_Benefits_Guide_2026_v1.pdf', 'page': 12, 'total_pages': 22, 'Author': '', 'CreationDate': "D:20260116002528+09'00'", 'Mod

In [16]:

from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    '''다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트:{context}

질문: {question}
'''
)

prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='다음 컨텍스트만 사용해 질문에 답하세요.\n컨텍스트:{context}\n\n질문: {question}\n')

In [20]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.output_parsers import StrOutputParser

# llm = OllamaLLM(model="gemma3:270m", base_url="http://localhost:11434") # 도커를 이용하고 있으므로 base_url을 지정해주어야 함
# llm = OllamaLLM(model="gemma3:1b")
llm = OllamaLLM(model="gemma4:e2b")

chain = prompt | llm | StrOutputParser()

In [21]:
response = chain.invoke({'context': results, 'question': query})

In [22]:
response

"제공된 컨텍스트에 따르면 결혼과 관련하여 받을 수 있는 혜택은 다음과 같습니다.\n\n**결혼 관련 주요 혜택:**\n\n*   **결혼 본인:** 휴가 5일, 경조금 100만원, 화환 지원\n*   **배우자 관련 (출산 관련):** 출산휴가 10일(유급), 출산 축하금 50만원, 과일 바구니 지원\n*   **부모/배우자 부모 관련:** 휴가 5일, 경조금 100만원, 장례용품 3단 조화 및 근조기 지원\n\n**기타 가족 관련 수당:**\n\n*   **부양 가족이 있는 자 배우자:** 5만원 (등본 제출 필수)\n*   **자녀 관련:** 자녀당 월 3만원 지원 (다만, 이는 '가족 수당' 항목에 대한 내용으로 보입니다.)"